#### Contexto

O decreto 8936 de 19 de dezembro de 2016 instituiu a Plataforma de Cidadania Digital, possibilitando abrir e acompanhar solicitações diretamente pela conta gov.br, inclusive por dispositivos móveis, sem a necessidade de atendimento presencial. 

O decreto, no Art. 1º, traz dois objetivos que orientam este projeto: 

"IV - simplificar as solicitações, a prestação e o acompanhamento dos serviços públicos, com foco na experiência do usuário;"

"V - dar transparência à execução e permitir o acompanhamento e o monitoramento dos serviços públicos;"

O dataset traz dados anonimizados dos atendimentos sobre dúvidas referentes à conta gov.br. 

Fonte: https://www.planalto.gov.br/ccivil_03/_ato2015-2018/2016/decreto/d8936.htm

#### Objetivo



Investigar os atendimentos de 2025, garantindo qualidade dos dados e consistência estatística de indicadores que irão para o painel de monitoramento. Os indicadores são: 

a) volume de solicitações; 

b) tempo mediano de atendimento.

#### Amostragem


Os dados raw passaram pelas fases de ETL em Python, conforme a seguir:
* Donwload no Portal de Dados Abertos dos dados de 2025, sendo 1 arquivo por mês;
* Leitura dos arquivos CSV e conversão para parquet;
* Envio em loop via API do BigQuery, com Load Job dos dados em uma tabela particionada por mês;
* Query amostral aleatória com 5000 registros por mês e export para CSV.

Optei por exportar uma amostra em SQL e trabalhar com Python por dois motivos: 

1) levar a fonte de dados para cloud, preparando o ambiente para o dashboard; 

2) economizar processamento computacional, tornando a análise exploratória mais fluida. Assim, saímos de 19,67 M linhas para 60 MIL registros, mantendo aleatoriedade e consistência estatística da amostra.

#### Descrição dos dados


* solicitacao - Identificador único do chamado (bigint)
* origem - Origem do chamado (string)
* categoria_solucao - Categoria do chamado (string) 
* datainicio - Data de criação do chamado (string)
* atendimentoinicio - Data de início do atendimento (string)
* encerramento - Data da solução do chamado (string)
* grupo - Time de atendimento responsável (string)
* situacao - Situação do encerramento (string)

Fonte - https://dados.gov.br/dados/conjuntos-dados/atendimentos-realizados-na-central-de-atendimento-da-conta-govbr

#### Imports

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from pandas.io.formats.style import Styler

#### Tratamento e limpeza dos dados

In [ ]:
# Ler arquivo
caminho_arquivo = r"C:\Users\Jorge Gabriel\OneDrive\Portfólio Analista de Dados\Central GOV BR - Indicadores de Atendimento\data\processed\amostra_bq-results-20260219-225733-1771542013682.csv"

df = pd.read_csv(filepath_or_buffer=caminho_arquivo,
                 encoding='utf-8'     
                 )

df.head()

In [ ]:
# Sumário dos dados 
df.info()

In [ ]:
# Renomear colunas no padrão camelCase
df2 = df.copy()
df2 = df.rename(columns={'categoria_solucao':'categoriaSolucao',
                   'datainicio': 'dataInicio',
                   'atendimentoinicio': 'atendimentoInicio',
                   })
df2.columns

#### Verificação de dados nulos

In [ ]:
df2[df2['categoriaSolucao'].isnull()]

In [ ]:
# Como a única solicitação null tem origem "Chat Online", verificaremos suas categorias de solução
df2.loc[df2['origem'] == 'Chat Online'].value_counts()

In [ ]:
# Valores nulos em categoriaSolucao foram imputados como "Chat Online > (N1)" quando origem = Chat Online, 
# pois 100% dos registros dessa origem são dessa categoria.
df2['categoriaSolucao'] = df2['categoriaSolucao'].fillna('Chat Online > (N1)')

In [ ]:
# Checar o sumário de dados após alteração
df2.info()

#### Verificar registros de atendimentos inválidos (encerramento <= atendimentoInicio)

In [ ]:
df2[pd.to_datetime(df2['encerramento']) <= pd.to_datetime(df2['atendimentoInicio'])]

#### Acrescentando Colunas

##### Duração do Atendimento

In [ ]:
# Duração do Atendimento = 'encerramento' - 'atendimentoInicio'
df2['atendimentoDuracao'] = pd.to_datetime(df2['encerramento']) - pd.to_datetime(df2['atendimentoInicio'])

##### Duração do Atendimento em Horas


In [ ]:
df2['atendimentoDuracaoHoras'] = df2['atendimentoDuracao'].dt.total_seconds() / 3600

##### Turnos de Atendimento

In [ ]:
df2['horaCriacao'] = pd.to_datetime(df2['dataInicio']).dt.hour
df2['horaInicio'] = pd.to_datetime(df2['atendimentoInicio']).dt.hour
df2['horaEncerramento'] = pd.to_datetime(df2['encerramento']).dt.hour

colunas_horas = ['horaCriacao', 'horaInicio', 'horaEncerramento']

for coluna in colunas_horas:
    condicoes = [
    (df2[coluna] >= 6) & (df2[coluna] < 12),
    (df2[coluna] >= 12) & (df2[coluna] < 18)
    ]
    
    valores = ['Manhã', 'Tarde']
    
    df2[f'turno{coluna}'] = np.select(condlist=condicoes, choicelist=valores, default='Noite')

##### Dias da semana

In [ ]:
# Convertendo dataInicio para datetime
df2['dataInicio'] = pd.to_datetime(df2['dataInicio'])

# Criando dicionário de tradução dos dias da semana
dias_ptbr = {
    'Monday': 'Segunda-feira',
    'Tuesday': 'Terça-feira',
    'Wednesday': 'Quarta-feira',
    'Thursday': 'Quinta-feira',
    'Friday': 'Sexta-feira',
    'Saturday': 'Sábado',
    'Sunday': 'Domingo'
}

# Criando uma coluna com dia da semana, a partir de dataInicio
df2['diaSemana'] = df2['dataInicio'].dt.day_name().map(dias_ptbr)

# Criando uma coluna com nº do dia na semana, a partir de dataInicio (0=Segunda, 6=Domingo)
df2['diaSemanaNum'] = df2['dataInicio'].dt.dayofweek

##### Datas de início e término do atendimento - Somente Data


In [ ]:
df2['diaInicio'] = pd.to_datetime(df2['dataInicio']).dt.normalize()
df2['diaEncerramento'] = pd.to_datetime(df2['encerramento']).dt.normalize()

df2.info()

#### Estatística Descritiva - Volume de Solicitações

##### 1) Qual é a origem de 80% dos atendimentos?



99,86% das solicitações têm origem em Facematch. 
Esta é a origem de acesso por reconhecimento facial e serve para Prova de Vida do INSS, elevar o nível da conta para Prata ou Ouro e Acesso Seguro. 

In [ ]:
# Freq. Absoluta
freq_origem = (df2.groupby(['origem'])[['solicitacao']]
               .count()
               .reset_index()
               .rename(columns={'solicitacao': 'freqAbs'})
               )

# Freq. Relativa
freq_origem['freqRelativa'] = freq_origem['freqAbs'] / freq_origem['freqAbs'].sum() * 100

# Tabela de frequências
freq_origem = freq_origem.sort_values(by='freqRelativa', ascending=False)
freq_origem.style.background_gradient(cmap='Blues',
                                      high=0.3, 
                                      low=0.1,
                                      subset=['freqRelativa', 'freqAbs'])



##### 2) Qual é o turno com maior volume de entrada de solicitações?


41,78% das solicitações são abertas no turno da tarde.

In [ ]:
# Frequência Absoluta
turno_inicio = (df2.groupby(['turnohoraInicio'])[['solicitacao']]
                .count()
                .reset_index()
                .rename(columns={'solicitacao': 'freqAbs'}))

# Frequência Relativa
turno_inicio['freqRelativa'] = turno_inicio['freqAbs'] / turno_inicio['freqAbs'].sum() * 100

# Ordenando por frequência absoluta
turno_inicio = turno_inicio.sort_values(by='freqAbs', ascending=False)

# Tabela de frequências
turno_inicio.style.background_gradient(cmap='Blues',
                                       low=0.1,
                                       high=0.3,
                                       subset=['freqAbs', 'freqRelativa'])



In [ ]:
# Ordenar dataframe
ordem = ['Manhã', 'Tarde', 'Noite']

turno_inicio_plot = turno_inicio.copy()

turno_inicio_plot['turnohoraInicio'] = pd.Categorical(
    turno_inicio_plot['turnohoraInicio'],
    categories=ordem,
    ordered=True
    )

turno_inicio_plot = turno_inicio_plot.sort_values('turnohoraInicio')

# Gráfico de barras de turnoinicioHora x freqRelativa
plt.figure(figsize=(10,4))
plt.bar(
    turno_inicio_plot['turnohoraInicio'],
    turno_inicio_plot['freqRelativa'],
    )
plt.ylim(0, 100)
plt.xlabel('Turno de Início do Atendimento', fontsize=12)
plt.ylabel('Frequência Relativa %',  fontsize=12)
plt.title('Distribuição de Atendimentos por Turnos',  fontsize=12)
plt.style.use('fivethirtyeight')
plt.show()

Em 2025, 40% das solicitações iniciaram no turno da tarde. Esse turno atingiu 43% do volume de solicitações nos meses de janeiro, abril e junho. Em geral, a proporção de solicitações por turno se manteve estável durante o ano. 

In [ ]:
turnohoraInicio_mes = (df2.groupby(['mes', 'turnohoraInicio'])[['solicitacao']]
                       .count()
                       .rename(columns={'solicitacao': 'freqAbs'}))

turnohoraInicio_mes['totalMesGeral'] = turnohoraInicio_mes.groupby(['mes'])[['freqAbs']].transform('sum') 

turnohoraInicio_mes['freqRelativa'] = turnohoraInicio_mes['freqAbs'] / turnohoraInicio_mes['totalMesGeral'] * 100

turnohoraInicio_mes = turnohoraInicio_mes.reset_index() 

turnohoraInicio_mes

In [ ]:
# copiar o dataset para uso no gráfico
turnoMes_plot = turnohoraInicio_mes.copy()

# criar uma lista que ordena os turnos de atendimento
ordem = ['Manhã', 'Tarde', 'Noite']

# Ordenar a variável por turno de início do atendimento
turnoMes_plot['turnohoraInicio'] = pd.Categorical(turnoMes_plot['turnohoraInicio'],
                                          categories=ordem,
                                          ordered=True)

# Gráfico de linhas de Frequência de atendimentos por mês e turno
plt.figure(figsize=(12,6))
sns.lineplot(data=turnoMes_plot,
             x='mes',
             y='freqRelativa', 
             hue='turnohoraInicio', 
             marker='o', 
             palette='deep')

plt.title('Frequência Relativa Mensal por Turno de atendimento', fontsize = 12)
plt.xlabel('Mês', fontsize = 10)
plt.ylabel('Frequência Relativa %', fontsize = 10)
plt.legend(title='Turno de Início', loc='upper right', fontsize = 10)
plt.ylim(0,100)
plt.style.use('fivethirtyeight')
plt.show()

##### 3) Qual categoria apresenta maior densidade de solicitações?

As categorias com maiores densidades foram "Triagem Login Únixo "Central de Serviços - Login Único > Triagem Login Único" e "Triagem Login Único", com 66,41% e 33,27%, respectivamente, somando 99,68% das solicitações do ano. Essas categorias representam o mesmo tipo de atendimento. Durante o ano houve mudança de nomenclatura.

In [ ]:
# Frequência Absoluta
freq_cat_solucao = (df2.groupby(['categoriaSolucao'])[['solicitacao']]
                    .count()
                    .rename(columns={'solicitacao': 'freqAbs'})
                    .reset_index())

# Frequência Relativa
freq_cat_solucao['freqRelativa'] = (freq_cat_solucao['freqAbs'] / freq_cat_solucao['freqAbs'].sum()) *100 

# Tabela de Frequências
freq_cat_solucao = freq_cat_solucao.sort_values(by='freqRelativa', ascending=False)
freq_cat_solucao.style.background_gradient(cmap='Blues',
                                           low=0.1,
                                           high=0.3,
                                           subset=['freqRelativa', 'freqAbs'])

##### 4) Quais são os grupos com maiores volumes de solicitações no ano?

O grupo com maior densidade de demanda foi Login Único – 1º e 2° Nível, correspondendo a 99,75% das solicitações do ano


In [ ]:
# Frequência Absoluta
freq_grupo = (df2.groupby(['grupo'])[['solicitacao']]
              .count()
              .rename(columns={'solicitacao': 'freqAbs'})
              .reset_index()
              )

# Frequência Relativa
freq_grupo['freqRelativa'] = freq_grupo['freqAbs'] / freq_grupo['freqAbs'].sum() * 100

# Tabela de Frequências
freq_grupo = freq_grupo.sort_values(by='freqRelativa', ascending=False)

freq_grupo.style.background_gradient(cmap='Blues',
                                     low=0.1,
                                     high=0.3,
                                     subset=['freqAbs', 'freqRelativa'])

##### 5) Em quanto ficou a taxa de aceitação das solicitações durante o ano de 2025?

A taxa de aceitação ficou em 73,88% em 2025

In [ ]:
# Frequência absoluta
freq_situacao = (df2.groupby(['situacao'])[['solicitacao']]
                 .count()
                 .rename(columns={'solicitacao': 'freqAbs'})
                 .reset_index()
                 )
# Frequência Relativa
freq_situacao['freqRelativa'] = freq_situacao['freqAbs'] / freq_situacao['freqAbs'].sum() * 100

# Tabela de Frequências
freq_situacao = freq_situacao.sort_values(by='freqRelativa', ascending=False)
freq_situacao.style.background_gradient(cmap='Blues',
                                        low=0.1,
                                        high=0.3,
                                        subset=['freqAbs', 'freqRelativa'])

99,8% das solicitações rejeitadas são da origem "Facematch", categoria de solução 'Triagem Login Único' e grupo "Login Único – 1º e 2° Nível"

In [ ]:
rejeicoes = (df2[df2['situacao'] == 'Solicitação rejeitada']
                         .groupby(['origem','categoriaSolucao', 'grupo'])[['solicitacao']]
                         .count()
                         .reset_index()
                         .rename(columns={'solicitacao': 'freqAbs'})
                         )

rejeicoes['freqRelativa'] = rejeicoes['freqAbs'] / rejeicoes['freqAbs'].sum() * 100

rejeicoes = rejeicoes.sort_values(by='freqRelativa', ascending=False)
rejeicoes.style.background_gradient(cmap='Blues',
                                        low=0.1,
                                        high=0.3,
                                        subset=['freqAbs', 'freqRelativa'])

In [ ]:
# Tabela de frequências de solicitacao x situacao x mes
situacao_mes = (df2.groupby(['mes', 'situacao'])[['solicitacao']]
                .count()
                .rename(columns={'solicitacao': 'freqAbs'})
                .reset_index())

situacao_mes['totalMesGeral'] = situacao_mes.groupby(['mes'])[['freqAbs']].transform('sum') 

situacao_mes['freqRelativa'] = situacao_mes['freqAbs'] / situacao_mes['totalMesGeral'] * 100


In [ ]:
# Evolução mensal das solicitações por situação
plt.figure(figsize=(14, 6)) 

sns.lineplot(data=situacao_mes,
             x='mes',
             y='freqRelativa', 
             hue='situacao', 
             marker='o', 
             palette='pastel'
             )

plt.title('Evolução mensal das solicitações por situação', fontsize=12)
plt.xlabel('Mês', fontsize=12)
plt.ylabel('% Frequência Relativa', fontsize=12)
plt.legend(title='Situação', loc='upper left', fontsize = 8)
plt.ylim(0, 100)  
plt.style.use('fivethirtyeight')
plt.show()

Análise com rolling de 28 dias - O comportamento das solicitações com meses normalizados em períodos de 28 dias não sofreu alterações significativas.

In [ ]:
# Criar uma agregação por dia
df28 = df2.copy()
df28['dataInicio'] = df28['dataInicio'].dt.date
df28['dataInicio'] = pd.to_datetime(df28['dataInicio'])

serie_diaria = (
    df28.groupby(['dataInicio', 'situacao'], observed=True)
    .agg(freqAbs=('situacao', 'count'))
    .sort_values(by='dataInicio')
    .reset_index())

In [ ]:
# Criando janela de 28 dias onde cada data_ref concentrará o resultado dos últimos 28 dias anteriores
rolling_28 = (
    serie_diaria
    .sort_values('dataInicio')      # ordenando por data
    .set_index('dataInicio')        # transformando a coluna de data em coluna de index para o pandas entender a ordenação 
    .groupby('situacao')['freqAbs'] # agrupando por 28 dias, somando as solicitações e contando os dias
    .rolling('28D')
    .agg(
        ['sum',
         'count'
        ]
    )
    .reset_index()
)

In [ ]:
rolling_28 = rolling_28.rename(columns={
    'dataInicio': 'data_ref',
    'sum': 'freqAbs',
    'count': 'dias_na_janela'
})

rolling_28 = rolling_28[['data_ref','situacao','freqAbs','dias_na_janela']]

In [ ]:
rolling_28['totalMesGeral'] = rolling_28.groupby(['data_ref'])[['freqAbs']].transform('sum') 

rolling_28['freqRelativa'] = rolling_28['freqAbs'] / rolling_28['totalMesGeral'] * 100

rolling_28

In [ ]:
plt.figure(figsize=(12, 6)) 

sns.lineplot(data=rolling_28,
             x='data_ref',
             y='freqRelativa', 
             hue='situacao', 
             marker='o', 
             palette='pastel'
             )

plt.title('Evolução mensal das solicitações x situação', fontsize=12)
plt.xlabel('Janela de 28 dias', fontsize=12)
plt.ylabel('Frequência Absoluta', fontsize=12)
plt.legend(title='Situação', loc='upper left', fontsize = 6)
plt.ylim(0,100)
plt.style.use('fivethirtyeight')
plt.show()

#### Estatísitcas Descritivas - Duração do Atendimento


##### Descrição da variável 'atendimentoDuracao'

In [ ]:
# Gerando tabela de estatísticas descritivas
stats_duracao = df2['atendimentoDuracao'].describe()

# Calculando amplitude e coeficiente de variação.
amp = df2['atendimentoDuracao'].max() - df2['atendimentoDuracao'].min()
cvar = df2['atendimentoDuracao'].std() / df2['atendimentoDuracao'].mean()

# Criando um DataFrame Pandas com amplitude e coeficiente de variação. 
extras = pd.DataFrame({
    'atendimentoDuracao': [amp, cvar]
    },
    index=['amplitude','coef_var'])

# Concatenando o DataFrame de medidas extras e o DataFrame gerado com .describe(). 
stats_duracao = pd.concat([stats_duracao.to_frame(), extras])

stats_duracao

##### 1) Como está a distribuição dos atendimentos em cada faixa de horas?
A grande maioria dos atendimentos está distribuida na faixa de duração de 0 a 10h.

In [ ]:
# plotar histograma com frequência de Duração do atendimento em horas
plt.figure(figsize=(12,6))
plt.hist(
    df2['atendimentoDuracaoHoras'],
    bins=20,
    range=(0,200),
    color='skyblue',
    edgecolor='black',
    )
x_ticks = np.arange(start=0,stop=220,step=10)
y_ticks = np.arange(start=0,stop=35000,step=5000)
plt.xticks(x_ticks)
plt.yticks(y_ticks)
plt.xlabel('Duração dos atendimentos em horas', fontsize=10)
plt.ylabel('Frequência', fontsize=10)
plt.title('Distribuição dos atendimentos por faixas de duração (horas)', fontsize=14)
plt.style.use('fivethirtyeight')
plt.show()

##### 1.1) Análise de outliers
A duração dos atendimentos tem alta variabilidade, com cauda longa à direita. Como não obtive acesso ao grau de complexidade dos atendimentos ou registros de erros na plataforma, optei por não remover outliers. 

Os valores extremos foram tratados como parte do comportamento do processo, sendo analisados separadamente como atendimentos de alta duração. Utilizei medidas estatísticas como mediana e percentis para melhor representação da distribuição.

In [ ]:
# Boxplot de atendimentoDuracao por mês

# Ordenar o df2 por mês
df2 = df2.sort_values('mes')

# plotar o gráfico
plt.figure(figsize=(16,8))
sns.boxplot(data= df2,
            x= 'mes',
            y= 'atendimentoDuracaoHoras')
plt.title('Duração de atendimento (h) por mês', fontsize=14)
plt.xlabel('Mês', fontsize=12)
plt.ylabel("Duração do atendimento (Horas)", fontsize=12)
plt.style.use('fivethirtyeight')
plt.show()

Os meses de agosto e do último trimestre apresentaram aumento significativo na mediana de duração dos atendimentos e na quantidade de outliers.

##### Segmentando atendimentos por faixas de duração
As faixas foram definidas por quartis de duração.

Rápido: até o quartil 0.25

Padrão: entre os quartis 0.25 e 0.75 

Longo: entre os quartis 0.75 e o limite superior de outliers, definido por q3 + 1.5 * iqr (zona de tolerância)

Crítico: Acima da zona de tolerância


In [ ]:
# Definir faixas de atendimentos pelos tempos de duração.
q1 = df2['atendimentoDuracao'].quantile(0.25).floor('s')
q3 = df2['atendimentoDuracao'].quantile(0.75).floor('s')

iqr = (q3 - q1).floor('s')
limite_outlier = (q3 + 1.5 * iqr).floor('s')

condicoes = [
    df2['atendimentoDuracao'] <= q1,
    (df2['atendimentoDuracao'] > q1) & (df2['atendimentoDuracao'] <= q3),
    (df2['atendimentoDuracao'] > q3) & (df2['atendimentoDuracao'] <= limite_outlier),
    df2['atendimentoDuracao'] > limite_outlier
]

valores = [f'01: Rápido <= {q1}', 
           f'02: Padrão > {q1} <= {q3}', 
           f'03: Longo > {q3} <= {limite_outlier}', 
           f'04: Crítico > {limite_outlier}']

df2['faixaDuracao'] = np.select(condlist=condicoes, choicelist=valores, default=f'04: Crítico > {limite_outlier}')

In [ ]:
# Gerar gráfico de barras com faixas de duração
faixas_plot = df2['faixaDuracao'].value_counts().reset_index()
faixas_plot = faixas_plot.sort_values(by='faixaDuracao', ascending=False)
plt.figure(figsize=(12,8))
plt.barh(
    faixas_plot['faixaDuracao'],
    faixas_plot['count']
    )
plt.style.use('fivethirtyeight')
plt.title('Frequência de atendimentos por faixas de duração', fontsize=12)
plt.xlabel('Frequência', fontsize=12)
plt.ylabel('Faixas de Duração', fontsize=12)
plt.show()

In [ ]:
# Tabela de frequências por faixas de duração

freq_faixa = (
    df2.groupby(['faixaDuracao'], observed=True)
    .agg(freqAbs=('faixaDuracao', 'count'))
    ).reset_index()

freq_faixa['freqRelativa'] = freq_faixa['freqAbs'] / freq_faixa['freqAbs'].sum() * 100

freq_faixa.style.background_gradient(cmap='Blues',
                                     low=0.1,
                                     high=0.3)

##### 2) Como está a variação do tempo de atendimento ao longo dos meses?

In [ ]:
# Calculando a mediana de Duração do atendimento (horas) x mes 
stats_mes = (df2.groupby(['mes'])[['atendimentoDuracaoHoras']].median()
             .rename(columns= {'atendimentoDuracaoHoras': 'medianaHoras'})
             )

stats_mes = stats_mes.reset_index()
stats_mes

In [ ]:
# Gráfico de linhas da mediana de 'atendimentoDuracaoHoras' x 'mes'
plt.figure(figsize = (12,6))
sns.lineplot(
    data=stats_mes.sort_values('mes', ascending=True),
    x='mes',
    y='medianaHoras'
)
plt.title('Mediana de Duração do Atendimento (h) por Mês', fontsize=14)
plt.xlabel('Mês', fontsize=12)
plt.ylabel('Mediana de Duração do Atendimento (horas)', fontsize=12)
plt.tight_layout()
plt.ylim(0,130)
plt.style.use('fivethirtyeight')
plt.show()

A duração dos atendimentos teve picos em março e agosto, podendo ser instabilidade operacional ou programa de governo que atraiu demanda fora do comum para a plataforma. Olharemos a evolução dia a dia neste mês.
 
Para o último trimestre, considerando que o aumento de duração dos atendimentos ocorreu progressivamente, podemos considerar comportamento normal da base. É válido detalhar em projetos futuros se esse aumento tem correlação com pagamentos de benefícios no final do ano, acesso a informações sobre 13º salário, regularização de pendências de documentos, entre outros serviços que os usuários buscam resolver no final do ano.

##### 2.1) Houve algum dia ou semana atípica que causou picos de duração dos atendimentos em março e agosto?

In [ ]:
df2_agosto = df2[df2['diaInicio'].dt.month == 8] 
stats_agosto = df2_agosto.groupby(['diaInicio'])['atendimentoDuracaoHoras'].agg(['median',
                                                                                  'mean',
                                                                                  'min',
                                                                                  'max',
                                                                                  'std'
                                                                                  ]
                                                                                  )

stats_agosto = stats_agosto.reset_index()

In [ ]:
# Gráfico de linhas da mediana de 'atendimentoDuracaoHoras' x 'dia de agosto'
plt.figure(figsize = (16,8))
sns.lineplot(
    data=stats_agosto.sort_values('diaInicio', ascending=True),
    x='diaInicio',
    y='mean'
)
plt.title('Mediana de Duração do Atendimento (h)) x Dia em Agosto/2025', fontsize=14)
plt.xlabel('Dia', fontsize=14)
plt.ylabel('Mediana de Duração do Atendimento (horas)', fontsize=14)
plt.ylim(0,200)
plt.style.use('fivethirtyeight')
plt.show()

Agosto - Podemos ver que o mês inteiro teve durações altas em comparação ao restante do ano. Contudo, entre os dias 9 e 16 de agosto o tempo de duração dos atendimentos subiu drasticamente. O atendimento que atingiu o tempo máximo nesse mês começou em 10/08/2025 e durou 524h.

In [ ]:
df2_marco = df2[df2['diaInicio'].dt.month == 3] 
stats_marco = df2_marco.groupby(['diaInicio'])['atendimentoDuracaoHoras'].agg(['median',
                                                                                  'mean',
                                                                                  'min',
                                                                                  'max',
                                                                                  'std'
                                                                                  ]
                                                                                  )

stats_marco = stats_marco.reset_index()

In [ ]:
# Gráfico de linhas da mediana de 'atendimentoDuracaoHoras' x 'dia de março'
plt.figure(figsize = (16,8))
sns.lineplot(
    data=stats_marco.sort_values('diaInicio', ascending=True),
    x='diaInicio',
    y='mean'
)
plt.title('Mediana de Duração do Atendimento (h) x Dia em Março/2025', fontsize=14)
plt.xlabel('Dia', fontsize=12)
plt.ylabel('Mediana de Duração do Atendimento (horas)', fontsize=12)
plt.ylim(0,100)
plt.style.use('fivethirtyeight')
plt.show()

Março - A última semana provocou o pico no tempo mediano de atendimento.

##### 3) Como está a duração dos atendimentos em cada situação de encerramento?

In [ ]:
stats_situacao = (
    df2
    .groupby(['mes', 'situacao'], observed=True)
    .agg(
        qtd_atend=('atendimentoDuracao', 'count'),
        mediana=('atendimentoDuracao', 'median'),
        media=('atendimentoDuracao', 'mean'),
        minimo=('atendimentoDuracao', 'min'),
        maximo=('atendimentoDuracao', 'max'),
        desvio_padrao=('atendimentoDuracao', 'std')
    )
    .reset_index()
    )

In [ ]:
stats_situacao['mediana(h)'] = stats_situacao['mediana'].dt.total_seconds()/60

plt.figure(figsize = (12,6))
sns.lineplot(
    data=stats_situacao.sort_values('mes', ascending=True),
    x='mes',
    y='mediana(h)',
    hue='situacao',
    legend=True
)
plt.legend(fontsize=8)
plt.title('Mediana de Duração do Atendimento (h) x Situação', fontsize=14)
plt.xlabel('Situação de Encerramento', fontsize=12)
plt.ylabel('Mediana de Duração do Atendimento (horas)', fontsize=12)
plt.tight_layout()
plt.style.use('fivethirtyeight')
plt.show()

##### 4) Qual é o turno com maior duração dos atendimentos?

In [ ]:
# Ordenando o dataset por turno

ordem = ['Manhã', 'Tarde', 'Noite']

turno_inicio = df2.copy()


turno_inicio['turnohoraInicio'] = pd.Categorical(
    turno_inicio['turnohoraInicio'],
    categories=ordem,
    ordered=True
    )

In [ ]:
# Calculando estatísticas

turno_stats = (
    turno_inicio
    .groupby(['turnohoraInicio'], observed=True)
    .agg(
        qtd_atend=('atendimentoDuracao', 'count'),
        mediana=('atendimentoDuracao', 'median'),
        media=('atendimentoDuracao', 'mean'),
        minimo=('atendimentoDuracao', 'min'),
        maximo=('atendimentoDuracao', 'max'),
        desvio_padrao=('atendimentoDuracao', 'std')
    )
    .reset_index()
    )


In [ ]:
# Convertendo mediana para horas
turno_stats['mediana(h)'] = turno_stats['mediana'].dt.total_seconds()/3600

In [ ]:
# Plotando Gráfico de Barras
plt.figure(figsize=(8,4))
plt.bar(
    turno_stats['turnohoraInicio'],
    turno_stats['mediana(h)']
)
plt.style.use('fivethirtyeight')
plt.xlabel('Turno', fontsize=12)
plt.ylabel('Mediana (horas)', fontsize=12)
plt.title('Mediana de Duração do Atendimento (h) por Turno',
          fontsize=12)
plt.style.use('fivethirtyeight')
plt.show()

In [ ]:
# turnohoraInicio x mes x atendimentoDuracao
stats_turno_mes = (
    df2
    .groupby(['mes', 'turnohoraInicio'])
    .agg(
        qtd_atend=('atendimentoDuracao', 'count'),
        mediana=('atendimentoDuracao', 'median'),
        media=('atendimentoDuracao', 'mean'),
        p90=('atendimentoDuracao', lambda x: x.quantile(0.9)),
        minimo=('atendimentoDuracao', 'min'),
        maximo=('atendimentoDuracao', 'max'),
        desvio_padrao=('atendimentoDuracao', 'std')
    )
    .reset_index()
)


In [ ]:
# Converter mediana para horas
stats_turno_mes['mediana(h)'] = stats_turno_mes['mediana'].dt.total_seconds()/3600

# Plotar Gráfico de Linhas
plt.figure(figsize=(12,6))
sns.lineplot(
    data=stats_turno_mes,
    x='mes',
    y='mediana(h)',
    hue='turnohoraInicio'
    )
plt.style.use('fivethirtyeight')
plt.title('Mediana de Duração dos Atendimentos x Mês', fontsize=12)
plt.legend(fontsize=10)

##### 5) Como está a duração dos atendimentos por dia da semana?

Os finais de semana têm maior mediana de tempo de atendimento, sendo acima de 10h de duração, com ênfase para os atendimentos iniciados aos Domingos, quando atingem 25h de duração mediana. 

In [ ]:
dia_semana = (
    df2
    .groupby(['diaSemana'])
    .agg(
        qtd_atend=('atendimentoDuracao', 'count'),
        mediana=('atendimentoDuracao', 'median'),
        media=('atendimentoDuracao', 'mean'),
        minimo=('atendimentoDuracao', 'min'),
        maximo=('atendimentoDuracao', 'max'),
        desvio_padrao=('atendimentoDuracao', 'std')
    )
    .reset_index()
)

dia_semana

In [ ]:
ordem_semana = ['Segunda-feira', 
                'Terça-feira', 
                'Quarta-feira', 
                'Quinta-feira', 
                'Sexta-feira', 
                'Sábado',
                'Domingo'
                ]

dia_semana_plot = dia_semana.copy()

dia_semana_plot['diaSemana'] = pd.Categorical(
    dia_semana_plot['diaSemana'],
    categories=ordem_semana,
    ordered=True
    )

dia_semana_plot = dia_semana_plot.sort_values(by='diaSemana')

plt.figure(figsize=(12,6))
plt.bar(
    dia_semana_plot['diaSemana'],
    dia_semana_plot['mediana'].dt.total_seconds()/3600 # mediana em horas
)
plt.style.use('fivethirtyeight')
plt.title('Mediana de Duração dos Atendimentos x Dia da Semana', fontsize=12)
plt.xlabel('Dia da Semana', fontsize=12)
plt.ylabel('Mediana em Horas', fontsize=12)
plt.ylim(0,30)
plt.show()




